# Designing self-reproducing RNAs with a Variational AutoEncoder

**Introduction to AI with applications in Science — MDC_42001_EP**
Session 11 · *Diversification of catalytic biomolecules* · lab, 1 h 30

---

This morning you saw that a statistical model fitted to **816 natural group I intron
sequences** can design new ribozymes carrying up to 60 mutations and still catalysing
their own splicing. This afternoon you run one of those models yourself.

The model is the **VAE** from Lambert, Opuu, Calvanese *et al.*, *Nature Communications*
**16**, 7836 (2025). The code and the trained weights are the authors' own
(Zenodo 10.5281/zenodo.16531362) — you are not reimplementing the paper, you are driving it.

**What you will do**

| | | ~min |
|---|---|---|
| 1 | Load the alignment and encode it | 10 |
| 2 | Two rulers: Hamming distance, and ViennaRNA secondary structure | 15 |
| 3 | The VAE: architecture, ELBO, pre-trained weights | 20 |
| 4 | Generate designs — and the one knob that matters | 15 |
| 5 | Structure-constrained decoding | 15 |
| 6 | Confront your designs with 24 220 measured ones | 15 |

There are **7 cells marked `# TODO`**. Everything else is given. Each section ends with a
`check_*()` cell that tells you whether you can move on.

**You will need:** `torch`, `ViennaRNA`, `numpy`, `pandas`, `matplotlib`.
```
pip install torch ViennaRNA numpy pandas matplotlib
```

In [ ]:
import sys, os, time, subprocess
import numpy as np

# ---------------------------------------------------------------- where we run
REPO = "https://github.com/philippenghe/X2A_RNA_VAE.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab detected - installing ViennaRNA (~20 s); torch is already here")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ViennaRNA"], check=False)

    if not os.path.isdir("data") and "CHANGE-ME" not in REPO:
        print("fetching the course material ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/td"], check=True)
        os.chdir("/content/td")

DATA = "data"

if not os.path.isdir(DATA):
    print("No data/ folder found.")
    print("  - on Colab: set REPO above to the course repository, or upload")
    print("    TD_VAE_RNA_design.zip and run  !unzip -qo TD_VAE_RNA_design.zip")
    print("  - locally: run this notebook from the folder that contains data/")
    raise SystemExit(1)

print()
print("python  ", sys.version.split()[0])
for name in ["torch", "RNA", "numpy", "pandas", "matplotlib", "scipy"]:
    try:
        m = __import__(name)
        print(f"{name:11s}", getattr(m, "__version__", "ok"))
    except ImportError:
        print(f"{name:11s} MISSING  ->  pip install " + {"RNA": "ViennaRNA"}.get(name, name))

print()
print("data    ", sorted(os.listdir(DATA)))
rng = np.random.default_rng(0)

---
## 1 · The training data

Everything the model knows about catalysis comes from one file: an alignment of group I
introns, one natural sequence per row, one aligned position per column. Positions that a
sequence does not have are written `-` (a gap).

Row 0 is the reference ribozyme, **Azoarcus**.

In [ ]:
NUC = ['A', 'G', 'U', 'C', '-']                 # the order used by the authors — keep it

msa = [l.strip() for l in open(f"{DATA}/azo_msa.seq") if l.strip()]
azo = open(f"{DATA}/azo.seq").read().strip()
L   = len(azo)

print(f"alignment : {len(msa)} sequences x {len(msa[0])} positions")
print(f"Azoarcus  : {L} nt")
print(f"row 0 is Azoarcus : {msa[0] == azo}")
print()
print(azo[:60], "...")

### TODO 1 — one-hot encoding

The network cannot read letters. Encode a sequence of length `L` as a flat vector of
length `L x 5`: for each position, a 1 in the slot of its nucleotide and 0 elsewhere,
in the order `NUC = ['A','G','U','C','-']`.

Return a `float32` array of shape `(L*5,)`.

In [ ]:
def one_hot(seq):
    """seq (str of length L) -> float32 array of shape (L*5,)"""
    # TODO 1
    raise NotImplementedError

In [ ]:
def check_1():
    v = one_hot(azo)
    assert v.shape == (L * 5,), f"expected shape ({L*5},), got {v.shape}"
    assert v.dtype == np.float32, f"expected float32, got {v.dtype}"
    assert v.sum() == L, "exactly one 1 per position"
    assert v.reshape(L, 5)[0, NUC.index(azo[0])] == 1, "wrong slot for position 0"
    print("one_hot OK — input layer will have", L * 5, "units")

check_1()

### What the alignment looks like

Before any model: how variable is each position? The **effective number of nucleotides**
`exp(S_i)` — where `S_i` is the Shannon entropy of the column — runs from 1 (one base only,
strictly conserved) to 4 (anything goes).

In [ ]:
import matplotlib.pyplot as plt

counts = np.zeros((L, 5))
for s in msa:
    for i, c in enumerate(s):
        counts[i, NUC.index(c)] += 1

freq   = counts[:, :4] / np.maximum(counts[:, :4].sum(1, keepdims=True), 1)   # drop gaps
gapf   = counts[:, 4] / len(msa)
with np.errstate(divide='ignore', invalid='ignore'):
    ent = -np.nansum(np.where(freq > 0, freq * np.log(freq), 0), axis=1)
neff = np.exp(ent)

fig, ax = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
ax[0].bar(range(L), neff, width=1.0, color="#2a5d8f")
ax[0].set_ylabel("eff. # nucleotides"); ax[0].set_ylim(1, 4)
ax[1].fill_between(range(L), gapf, color="#c0504d", alpha=.6)
ax[1].set_ylabel("gap fraction"); ax[1].set_xlabel("alignment position")
plt.tight_layout(); plt.show()

print(f"strictly conserved positions (eff < 1.2): {(neff < 1.2).sum()} / {L}")
print(f"freely variable positions  (eff > 3.0): {(neff > 3.0).sum()} / {L}")

**Look at the top panel.** Conservation is not uniform: blocks of near-invariant positions
alternate with stretches that tolerate anything. A model that only reproduces this profile
— one frequency per position, positions independent — is the `PRO` baseline from the
lecture, and half of its designs are dead by 5 mutations. The VAE is fitted to the same
data; the difference is what else it can represent.

---
## 2 · Two rulers

To judge a design we need to measure two different things.

**How far has it travelled?** — the Hamming distance to Azoarcus.
**Is it still the same molecule?** — whether it still folds into the same secondary structure.

### TODO 2 — Hamming distance

Count the positions at which two sequences of equal length differ.

In [ ]:
def hamming(a, b):
    """number of differing positions between two equal-length sequences"""
    # TODO 2
    raise NotImplementedError

In [ ]:
def check_2():
    assert hamming("AUGC", "AUGC") == 0
    assert hamming("AUGCAUGC", "AUACAUAC") == 2
    assert hamming(azo, azo) == 0
    d = [hamming(azo, s.replace('-', 'A')) for s in msa[1:200]]
    print(f"hamming OK — natural homologues sit {min(d)}-{max(d)} mutations from Azoarcus")

check_2()

### The target secondary structure

`target.str` is the secondary structure of Azoarcus in dot-bracket notation, taken from the
crystal structure. Round brackets are ordinary nested base pairs; **square brackets are a
pseudoknot** — a pair that crosses another pair.

ViennaRNA's folding algorithm is a dynamic program over *nested* structures: it cannot
represent a pseudoknot at all. So before comparing anything with Vienna we strip the
pseudoknot out. That is a real limitation, not a detail — the authors note it in the paper
(NUPACK handles pseudoknots but is far too slow to screen millions of candidates).

In [ ]:
def paired_positions(struct):
    """dot-bracket -> list of (i, j) pairs, 0-indexed. Handles () and []."""
    reg, pk, pairs = [], [], []
    for i, c in enumerate(struct):
        if   c == '(': reg.append(i)
        elif c == '[': pk.append(i)
        elif c == ')': pairs.append((reg.pop(), i))
        elif c == ']': pairs.append((pk.pop(), i))
    return pairs

target = open(f"{DATA}/target.str").read().strip()
target_nested = ''.join('.' if c in '[]' else c for c in target)

pairs_all    = paired_positions(target)
pairs_nested = paired_positions(target_nested)

print(f"target length          : {len(target)}")
print(f"base pairs in total    : {len(pairs_all)}   (of which pseudoknot: {target.count('[')})")
print(f"base pairs Vienna sees : {len(pairs_nested)}")
print(f"unpaired positions     : {target.count('.')}")
print()
print(target[:80])
print(target_nested[:80])


def bp_distance(struct_a, struct_b):
    """number of base pairs present in one structure and not the other"""
    a, b = set(paired_positions(struct_a)), set(paired_positions(struct_b))
    return len(a ^ b)


assert bp_distance(target, target) == 0
assert bp_distance(target, target_nested) == target.count('[')   # the 6 pseudoknot pairs

### TODO 3 — folding a design and comparing it to the target

`RNA.fold(seq)` returns `(structure, free_energy)` for the minimum-free-energy fold of a
sequence. You already have `bp_distance`.

Write `structure_distance(seq)`: fold the sequence, and return the base-pair distance
between that fold and `target_nested`. Lower is better.

In [ ]:
import RNA

def structure_distance(seq):
    """base-pair distance between the MFE fold of seq and the target structure"""
    # TODO 3  —  RNA.fold(seq) -> (structure, energy);  bp_distance(a, b) -> int
    raise NotImplementedError

In [ ]:
def check_3():
    ss, mfe = RNA.fold(azo)
    d = structure_distance(azo)
    print(f"Azoarcus MFE energy        : {mfe:.1f} kcal/mol")
    print(f"Azoarcus MFE vs target     : {d} base pairs apart")
    print()
    print("target :", target_nested[:80])
    print("MFE    :", ss[:80])
    scram = ''.join(rng.permutation(list(azo)))
    print()
    print(f"a shuffled sequence        : {structure_distance(scram)} base pairs apart")
    assert isinstance(d, (int, np.integer)), "should return an integer"
    print("\nstructure_distance OK")

check_3()

**The wild type is not at distance 0 — and that is expected.** The target comes from a
crystal structure; `RNA.fold` returns the single lowest-energy nested fold under a
thermodynamic model. The two disagree, partly because of the pseudoknot we just removed.

So do not read this number as truth. Read it as a **relative** ruler: a design that folds
further from the target than Azoarcus does is a design that has lost something. That is all
we will ask of it — and in section 6 you will check, against real measurements, whether
that is enough to be useful.

---
## 3 · The VAE

Below is the authors' architecture, unchanged. An encoder maps a one-hot sequence
(`193 x 5 = 965` inputs) through two hidden layers of 512 to a **latent code of dimension
128**; a decoder maps it back to 965 numbers, read as a probability distribution over the
5 symbols at each of the 193 positions.

Two lines are missing.

### TODO 4 and 5 — the reparametrisation and the ELBO

**TODO 4.** The encoder returns `mu` and `sig`. To sample the latent code we need
`z = mu + exp(sig) * eps` with `eps` standard normal — written this way so that the
gradient still flows through `mu` and `sig`.

*(This is the authors' convention. If `sig` were the log-variance the textbook form would
be `exp(sig/2)`; it is worth noticing the discrepancy, and it changes nothing below because
we use their trained weights.)*

**TODO 5.** The ELBO is the reconstruction term plus the KL term, averaged over the batch:
`(log_lk + kld).mean()`.

In [ ]:
import torch
from torch import nn

class Block(nn.Module):
    def __init__(self, d_in, d_out, act="relu"):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(d_in, d_out),
            nn.Dropout(p=0.1),
            nn.ReLU() if act == "relu" else nn.Identity(),
        )
    def forward(self, x):
        return self.block(x)


class VAE_seq(nn.Module):
    def __init__(self, len_seq, nb_state=5, nb_hid=2, dim_hid=2**9, dim_lat=2**7):
        super().__init__()
        dim_node = len_seq * nb_state
        self.dim_lat, self.nb_state = dim_lat, nb_state

        self.enc_node = nn.ModuleList([Block(dim_node, dim_hid)])
        for _ in range(nb_hid - 1):
            self.enc_node += [Block(dim_hid, dim_hid)]
        self.enc_node += [Block(dim_hid, dim_lat, act="id")]
        self.enc_si = Block(dim_hid, dim_lat, act="id")

        self.dec_node = nn.ModuleList([Block(dim_lat, dim_hid)])
        for _ in range(nb_hid - 1):
            self.dec_node += [Block(dim_hid, dim_hid)]
        self.dec_node += [Block(dim_hid, dim_node, act="id")]

    def encoder(self, x):
        for lin in self.enc_node[:-1]:
            x = lin(x)
        return self.enc_node[-1](x), self.enc_si(x)          # mu, sig

    def decoder(self, z):
        for lin in self.dec_node[:-1]:
            z = lin(z)
        z = self.dec_node[-1](z)
        z = z.view(z.shape[:-1] + (-1, self.nb_state))        # (..., L, 5)
        return torch.log_softmax(z, dim=-1)

    def forward(self, x):
        mu, sig = self.encoder(x)
        eps = torch.randn_like(sig)
        z = ...                                                          # TODO 4
        log_p  = self.decoder(z).flatten(start_dim=-2)
        kld    = (0.5 * (1 + sig - mu**2 - torch.exp(sig))).sum(-1)
        log_lk = (log_p * x).sum(-1)
        return ...                                                       # TODO 5

In [ ]:
model = VAE_seq(L)
state = torch.load(f"{DATA}/vae_weights.dat", map_location="cpu")
model.load_state_dict(state)
model.eval()          # <- switches Dropout off. Forget this and every design is noise.

n_par = sum(p.numel() for p in model.parameters())
print(f"parameters loaded : {n_par:,}")
for k, v in list(state.items())[:4]:
    print(f"  {k:28s} {tuple(v.shape)}")
print("  ...")

### Checkpoint: does the model know Azoarcus?

Encode the wild type, take the **mean** of its latent distribution (no noise), decode, and
read off the most likely base at each position. If the model has learnt this family at all,
it should give the wild type back exactly.

In [ ]:
@torch.no_grad()
def decode_free(logp):
    """(n, L, 5) log-probabilities -> (n, L) nucleotide indices, gaps excluded"""
    return logp[:, :, :4].argmax(-1)

def to_seq(idx):
    return [''.join(NUC[i] for i in row) for row in np.asarray(idx)]


def check_4():
    x = torch.tensor(one_hot(azo))
    with torch.no_grad():
        mu, sig = model.encoder(x)
        rec = to_seq(decode_free(model.decoder(mu[None, :])))[0]
    d = hamming(rec, azo)
    print(f"latent code : dimension {mu.shape[0]}, norm {mu.norm():.2f}")
    print(f"reconstruction of Azoarcus from the latent mean : {d} mutations")
    assert d == 0, ("the wild type should come back exactly — check TODO 4/5, "
                    "and check that model.eval() was called")
    print("\nVAE OK")

check_4()

*Optional (2 min).* Watch the ELBO improve from scratch. This is the authors' training loop,
cut from 10 000 steps to 150 — enough to see the loss fall, nowhere near enough to be usable.
Skip it if you are short of time.

In [ ]:
RUN_TRAINING = False          # flip to True if you want to see it

if RUN_TRAINING:
    fresh = VAE_seq(L); fresh.train()
    batch = torch.tensor(np.stack([one_hot(s) for s in msa]))
    opt = torch.optim.Adam(fresh.parameters(), weight_decay=0.01)
    t0 = time.time()
    for step in range(150):
        opt.zero_grad()
        loss = -fresh(batch)
        loss.backward(); opt.step()
        if step % 25 == 0:
            print(f"step {step:4d}   -ELBO = {loss.item():10.1f}")
    print(f"({time.time()-t0:.0f} s for 150 steps; the paper used 10 000)")

---
## 4 · Generating designs, and the one knob that matters

To generate, we do not encode anything new. We take the latent code of Azoarcus, **add
noise around it**, and decode. The size of that noise is the parameter `scale` — the
variance of the Gaussian we draw from:

> z ~ N( mu(Azoarcus) , scale x I )

Small `scale`: we stay next to the wild type and almost everything works. Large `scale`: we
travel, and more of what we make is broken. This is the exploration–exploitation trade-off
from the lecture, and here it is a single number you control.

### TODO 6 — the sampling line

Draw `n` latent codes around `mu`. The covariance is `scale * I`, so the standard deviation
per dimension is `sqrt(scale)`.

In [ ]:
@torch.no_grad()
def generate(n, scale, ref=azo, seed=None):
    """n designs sampled around the latent code of `ref`. Returns (n, L, 5) log-probs."""
    if seed is not None:
        torch.manual_seed(seed)
    mu, _ = model.encoder(torch.tensor(one_hot(ref)))
    z = ...                                                              # TODO 6
    return model.decoder(z)

In [ ]:
logp = generate(5, scale=1.0, seed=0)
for s in to_seq(decode_free(logp)):
    print(f"{hamming(s, azo):3d} mut   {s[:70]}...")
print(f"\nAzoarcus   {azo[:70]}...")

### Sweep the knob

For each `scale`: how far do the designs go, how different are they **from each other**,
and what does Vienna say about their structure?

Folding 60 sequences per point takes a few seconds — be patient.

In [ ]:
SCALES = [0.1, 0.3, 0.5, 1.0, 1.5, 2.0, 3.0]
N      = 60
rows   = []

for sc in SCALES:
    seqs = to_seq(decode_free(generate(N, sc, seed=1)))
    dwt  = [hamming(s, azo) for s in seqs]
    dpw  = [hamming(seqs[i], seqs[j]) for i in range(20) for j in range(i + 1, 20)]
    dstr = [structure_distance(s) for s in seqs]
    rows.append((sc, np.mean(dwt), np.mean(dpw), len(set(seqs)) / N, np.mean(dstr)))
    print(f"scale {sc:4.1f}  |  {np.mean(dwt):5.1f} mut to WT  |  {np.mean(dpw):5.1f} between designs"
          f"  |  {len(set(seqs))/N:4.0%} unique  |  structure distance {np.mean(dstr):5.1f}")

rows = np.array(rows)

In [ ]:
d_wt_struct = structure_distance(azo)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(rows[:, 0], rows[:, 1], 'o-', label="to Azoarcus")
ax[0].plot(rows[:, 0], rows[:, 2], 's--', label="between designs")
ax[0].set_xlabel("scale"); ax[0].set_ylabel("Hamming distance"); ax[0].legend(); ax[0].set_title("how far we travel")
ax[1].plot(rows[:, 0], rows[:, 4], 'o-', color="#c0504d")
ax[1].axhline(d_wt_struct, ls=':', color="k", label="Azoarcus itself")
ax[1].set_xlabel("scale"); ax[1].set_ylabel("bp distance to target"); ax[1].legend()
ax[1].set_title("what it costs structurally")
plt.tight_layout(); plt.show()

**Questions to settle before moving on.**

1. At which `scale` do the designs stop being copies of each other?
2. The two curves on the left do not grow at the same rate. Why is the distance *between*
   designs larger than the distance to the wild type?
3. On the right, structure distance climbs with `scale`. Some of that is real damage — but
   part of it is an artefact of taking the single MFE fold of a sequence we have pushed far
   from anything natural. How would you tell the two apart?
4. The paper's own script ships with `scale = 0.1`. Given the first column, what does that
   tell you about how they used this model?

*Optional.* If `ipywidgets` is installed, this gives you a slider instead of a loop.

In [ ]:
try:
    from ipywidgets import interact, FloatSlider

    def explore(scale=1.0):
        seqs = to_seq(decode_free(generate(12, scale, seed=2)))
        for s in seqs[:6]:
            print(f"{hamming(s, azo):3d} mut   {s[:64]}...")
        print(f"\nmean {np.mean([hamming(s, azo) for s in seqs]):.1f} mutations")

    interact(explore, scale=FloatSlider(min=0.05, max=4.0, step=0.05, value=1.0))
except ImportError:
    print("ipywidgets not installed — skip this cell, the loop above does the same job")

---
## 5 · Decoding with the structure in mind

So far we read the decoder greedily: at each position, the most likely base. Nothing stops
it from breaking a base pair — picking a G where the partner position has an A.

The authors do something smarter, and it costs almost nothing. For every pair `(i, j)` in
the target structure, instead of choosing position `i` and position `j` independently,
**choose the canonical pair that the decoder likes best**:

> argmax over the 6 canonical pairs  of  logp[i, base_i] + logp[j, base_j]

This is `compatible_seq` in the authors' `vae.py`.

### TODO 7 — constrained decoding

Start from the free decoding, then overwrite the paired positions.
`CAN_I[k]`, `CAN_J[k]` are the nucleotide indices of the k-th canonical pair.

In [ ]:
CAN   = [('A','U'), ('U','A'), ('G','C'), ('C','G'), ('G','U'), ('U','G')]
CAN_I = torch.tensor([NUC.index(a) for a, b in CAN])
CAN_J = torch.tensor([NUC.index(b) for a, b in CAN])

@torch.no_grad()
def decode_with_pairs(logp, pairs):
    """(n, L, 5) log-probs -> (n, L) indices, with every pair in `pairs` canonical"""
    idx = logp[:, :, :4].argmax(-1).clone()
    for i, j in pairs:
        # TODO 7 — pick the best canonical pair for (i, j) and write both positions
        pass
    return idx

In [ ]:
def canonical_fraction(seq, pairs):
    return sum((seq[i], seq[j]) in CAN for i, j in pairs) / len(pairs)


def check_5():
    print(f"Azoarcus is canonical at {canonical_fraction(azo, pairs_all):.0%} of its target pairs\n")
    print(" scale |     free decoding      |  structure-constrained ")
    print("       |  mut   canon   struct  |  mut   canon   struct  ")
    for sc in [0.5, 1.0, 2.0, 3.0]:
        lp = generate(40, sc, seed=3)
        free = to_seq(decode_free(lp))
        cons = to_seq(decode_with_pairs(lp, pairs_all))
        f = (np.mean([hamming(s, azo) for s in free]),
             np.mean([canonical_fraction(s, pairs_all) for s in free]),
             np.mean([structure_distance(s) for s in free]))
        c = (np.mean([hamming(s, azo) for s in cons]),
             np.mean([canonical_fraction(s, pairs_all) for s in cons]),
             np.mean([structure_distance(s) for s in cons]))
        print(f" {sc:5.1f} | {f[0]:5.1f}  {f[1]:5.0%}  {f[2]:6.1f}  | {c[0]:5.1f}  {c[1]:5.0%}  {c[2]:6.1f}")
    lp = generate(20, 2.0, seed=9)
    cons = to_seq(decode_with_pairs(lp, pairs_all))
    assert all(canonical_fraction(s, pairs_all) == 1.0 for s in cons), \
        "constrained decoding must give 100% canonical pairs"
    print("\ndecode_with_pairs OK")

check_5()

**Read the table.** Constrained decoding buys 100 % canonical pairing *for free* — the
designs travel just as far. But look at the structure-distance column: forcing the pairs to
be chemically possible is **not** the same as making the molecule fold into the target. The
constraint is local; folding is global.

That gap is the whole reason the paper also needed a thermodynamic score, and why
`DCA-SB` beats `DCA` alone.

---
## 6 · Confronting 24 220 measured designs

`design_db_T37C.csv` holds every design the authors synthesised and assayed at 37 °C: the
model that proposed it, its measured **activity** (log of the enrichment after the
self-splicing reaction, 0 for Azoarcus), and the authors' own DCA and structure scores.

A design counts as **active** above the noise threshold, `activity > -2.76`.

Note the sequences are 197 nt: a 3-nucleotide prefix and a trailing G were added for the
assay. The 193-nt core is `SEQ[3:196]`.

In [ ]:
import pandas as pd

db = pd.read_csv(f"{DATA}/design_db_T37C.csv")
for c in ["ACTIVITY", "DCA", "SB"]:
    db[c] = pd.to_numeric(db[c], errors="coerce")
db["CORE"] = db.SEQ.str[3:196]
db["NMUT"] = db.CORE.map(lambda s: hamming(s, azo))
db["ACTIVE"] = db.ACTIVITY > -2.76

print(db.MODEL.value_counts().to_string())
print(f"\nmeasured activity available for {db.ACTIVITY.notna().sum():,} of {len(db):,} designs")

In [ ]:
bins = [0, 10, 20, 30, 40, 50, 60, 200]
d = db.dropna(subset=["ACTIVITY"])
piv = (d.groupby(["MODEL", pd.cut(d.NMUT, bins)], observed=True)
         .ACTIVE.mean().unstack())

order = ["RDM", "profile", "SB", "BPR", "BPR-3D", "SB-3D", "VAE", "DCA_T=1", "DCA T=$0.3$", "DCA-SB"]
piv = piv.reindex([m for m in order if m in piv.index])

fig, ax = plt.subplots(figsize=(8, 4.5))
for m, row in piv.iterrows():
    style = dict(lw=2.5, marker='o') if m == "VAE" else dict(lw=1.2, alpha=.65, marker='.')
    ax.plot([b.mid for b in row.index], row.values, label=m, **style)
ax.set_xlabel("mutations to Azoarcus"); ax.set_ylabel("fraction active")
ax.legend(fontsize=8, ncol=2); ax.set_title("Reproducing Figure 2b of the paper")
plt.tight_layout(); plt.show()

print(piv.round(2).to_string())

This is the figure from the lecture, rebuilt from the raw measurements. The VAE (thick line)
holds up to about 20 mutations and then falls away; `DCA T=0.3` and `DCA-SB` go much
further. **Your generated designs live on the same axis.**

*(The gap in the VAE line between 40 and 50 mutations is not a bug: no VAE design in that
bin came back with a usable read. Real data has holes.)*

### Does the structure check actually predict activity?

You now have a filter — Vienna structure distance — and 24 220 designs whose fate is known.
Test the filter on the model you are driving. Compare only designs at a **comparable number
of mutations**, otherwise you are just rediscovering that distant designs fail.

In [ ]:
sel = db[(db.MODEL == "VAE") & db.ACTIVITY.notna() & db.NMUT.between(20, 50)].copy()
print(f"{len(sel)} VAE designs between 20 and 50 mutations, {sel.ACTIVE.mean():.0%} of them active")

fig, ax = plt.subplots(figsize=(5, 3.8))
ax.boxplot([sel[~sel.ACTIVE].SB.dropna(), sel[sel.ACTIVE].SB.dropna()], widths=.55)
ax.set_xticks([1, 2]); ax.set_xticklabels(["inactive", "active"])
ax.set_ylabel("structure score (lower = closer to target)")
plt.tight_layout(); plt.show()

from scipy.stats import mannwhitneyu
a, b = sel[sel.ACTIVE].SB.dropna(), sel[~sel.ACTIVE].SB.dropna()
print(f"median structure score   active {a.median():.2f}   inactive {b.median():.2f}")
print(f"Mann-Whitney p = {mannwhitneyu(a, b).pvalue:.1e}")

The separation is real but modest: the distributions overlap heavily. A structure score is
a **filter, not an oracle** — it will enrich your batch for active designs without
guaranteeing any single one. That is precisely why the paper needed 24 220 measurements and
not a clever score.

*(The `SB` column is the authors' score — the Vienna ensemble defect. Yours, from `RNA.fold`,
is cruder. If you have time, check how well the two agree on a few hundred designs.)*

---
## 7 · Challenge

Produce a batch of **100 designs** that goes as far as possible from Azoarcus while keeping
its structure. You choose: the `scale`, the decoding, and any filter you like on top.

The score is deliberately crude, and arguing about it is part of the exercise:

> **score = median number of mutations x fraction of the batch whose structure distance is no worse than Azoarcus + 10**

Run `submit("your name", seqs)`. The leaderboard is shared within the notebook session.

In [ ]:
LEADERBOARD = []

def submit(name, seqs, verbose=True):
    assert len(seqs) == 100, f"submit exactly 100 designs, got {len(seqs)}"
    assert all(len(s) == L for s in seqs), f"every design must be {L} nt"
    assert all(set(s) <= set("AGUC") for s in seqs), "A, G, U, C only — no gaps"
    budget = structure_distance(azo) + 10
    mut  = [hamming(s, azo) for s in seqs]
    keep = [structure_distance(s) <= budget for s in seqs]
    score = float(np.median(mut) * np.mean(keep))
    LEADERBOARD.append((score, name, float(np.median(mut)), float(np.mean(keep)),
                        len(set(seqs))))
    if verbose:
        print(f"{name}: median {np.median(mut):.0f} mutations, "
              f"{np.mean(keep):.0%} pass the structure filter, {len(set(seqs))} distinct"
              f"  ->  score {score:.1f}")
    return score


def leaderboard():
    print(f"{'rank':>4}  {'score':>6}  {'median mut':>10}  {'pass':>5}  {'uniq':>5}  name")
    for r, (s, n, m, k, u) in enumerate(sorted(LEADERBOARD, reverse=True), 1):
        print(f"{r:>4}  {s:6.1f}  {m:10.0f}  {k:5.0%}  {u:5d}  {n}")


# a baseline to beat
base = to_seq(decode_with_pairs(generate(100, 1.0, seed=42), pairs_all))
submit("baseline (scale=1, constrained)", base)

In [ ]:
# TODO — your turn.
#
#   ideas:
#     * push the scale up and filter hard on structure_distance
#     * generate 1000 and keep the best 100
#     * decode with pairs_all vs pairs_nested — does the pseudoknot constraint help?
#     * seed the generation from a natural homologue instead of Azoarcus:
#       generate(n, scale, ref=msa[17].replace('-', 'A'))
#
my_designs = ...

# submit("my name", my_designs)
# leaderboard()

---
## What you should take away

- A generative model trained on **816 sequences** reconstructs the wild type exactly and
  proposes new ones around it. The quantity of data is small; the structure in it is not.
- **One parameter** — the width of the latent sampling — moves you along the
  exploration–exploitation trade-off. Everything else in the pipeline is downstream of it.
- A **physical check** (does it still fold?) is cheap, independent of the training data, and
  it genuinely enriches for activity — without ever being decisive.
- And the honest part: the real curve in section 6 exists because somebody synthesised and
  sequenced 24 220 molecules. No score in this notebook replaces that.

### Going further

The same Zenodo archive contains `fully_connected.dca` — the trained DCA model, 464 164
coupling terms in a plain text file (`i j a b value`). Scoring a sequence with it is six
lines. Compare the DCA score and the VAE's log-likelihood on the same designs: do they
agree about which ones are plausible?